In [ ]:
import requests
import json
import time
import zipfile
import os


USERNAME = "karastatham"
PASSWORD = "*vSH4P#2YvyBdHJW"

In [ ]:
import os
import json
import time
import zipfile
import requests
import getpass
import geopandas as gpd
from tqdm import tqdm

In [ ]:
# -----------------------------
# User settings
# -----------------------------

SHAPEFILE = "Fixed_CA_Cities/Fixed_CA_Cities.shp"

OUTPUT_DIR = "AppEEARS_Annual_Output"

YEARS = range(2018, 2026)

API = "https://appeears.earthdatacloud.nasa.gov/api/"

# Example:
# ECO_L2T_LSTE.002
PRODUCT = "ECO_L2T_LSTE.002"
os.makedirs(OUTPUT_DIR, exist_ok=True)


In [ ]:
layers = requests.get(
    f"{API}product/{PRODUCT}"
).json()

available_layers = list(layers.keys())

print(available_layers)


wanted = {
    "LST",
    "QC",
    "cloud",
    "water"
}

# Keep only requested layers that actually exist
LAYERS = [
    layer for layer in available_layers
    if layer in wanted
]

print(LAYERS)


# Build AppEEARS layer request
appeears_layers = [
    {
        "product": PRODUCT,
        "layer": layer
    }
    for layer in LAYERS
]

print(appeears_layers)

In [ ]:
username = "karastatham"
password = "*vSH4P#2YvyBdHJW"

# username = getpass.getpass(
#     "NASA Earthdata username: "
# )
#
# password = getpass.getpass(
#     "NASA Earthdata password: "
# )


response = requests.post(
    f"{API}login",
    auth=(username,password)
)


response.raise_for_status()


token = response.json()["token"]


headers = {
    "Authorization": f"Bearer {token}"
}


print("Login successful")

In [ ]:
gdf = gpd.read_file(SHAPEFILE)


# AppEEARS works best with WGS84
if gdf.crs != "EPSG:4326":
    gdf = gdf.to_crs(4326)


region_geojson = json.loads(
    gdf.to_json()
)


print(
    "Features:",
    len(region_geojson["features"])
)

In [ ]:
from datetime import datetime, timedelta


def submit_appeears_request(start_date, end_date):

    task = {

        "task_type": "area",

        "task_name":
            f"ECO_L2T_LSTE_{start_date}_{end_date}",


        "params": {

            "dates": [
                {
                    "startDate": start_date,
                    "endDate": end_date
                }
            ],

            "layers": [
                {
                    "product": PRODUCT,
                    "layer": layer
                }
                for layer in LAYERS
            ],

            "output": {

                "format": {
                    "type": "geotiff"
                },

                "projection": "geographic"

            },

            "geo": region_geojson

        }

    }


    print("REQUEST DATES:")
    print(task["params"]["dates"])

    print("REQUEST LAYERS:")
    print(task["params"]["layers"])


    r = requests.post(
        f"{API}task",
        json=task,
        headers=headers
    )


    print("STATUS:", r.status_code)

    if r.status_code != 200:
        print("RESPONSE:", r.text)


    r.raise_for_status()

    return r.json()["task_id"]

In [ ]:
def wait_for_task(task_id):

    while True:

        status = requests.get(
            f"{API}task/{task_id}",
            headers=headers
        ).json()


        state = status["status"]


        if state == "done":
            return True


        if state == "error":
            raise Exception(status)


        time.sleep(30)

In [ ]:
from concurrent.futures import ThreadPoolExecutor, as_completed
from tqdm import tqdm
import shutil
from io import BytesIO

def download_bundle_to_zip(task_id, zip_file, year):

    bundle = requests.get(
        f"{API}bundle/{task_id}",
        headers=headers
    ).json()


    files = bundle["files"]


    for file in tqdm(
        files,
        desc=f"Downloading {year}",
        leave=False
    ):

        file_id = file["file_id"]

        filename = os.path.basename(
            file["file_name"]
        )


        r = requests.get(
            f"{API}bundle/{task_id}/{file_id}",
            headers=headers,
            stream=True
        )


        r.raise_for_status()


        # write directly into zip
        with zip_file.open(filename, "w") as z:

            for chunk in r.iter_content(
                chunk_size=1024*1024
            ):
                if chunk:
                    z.write(chunk)

In [ ]:
def zip_year(year):

    folder = os.path.join(
        OUTPUT_DIR,
        str(year)
    )


    zip_path = os.path.join(
        OUTPUT_DIR,
        f"{year}.zip"
    )


    with zipfile.ZipFile(
        zip_path,
        "w",
        zipfile.ZIP_DEFLATED
    ) as z:


        for root,dirs,files in os.walk(folder):

            for file in files:

                filepath = os.path.join(
                    root,
                    file
                )

                z.write(
                    filepath,
                    arcname=file
                )


    print(
        "Created:",
        zip_path
    )

In [ ]:
def weekly_ranges(year):

    start = datetime(year, 1, 1)
    end = datetime(year, 12, 31)

    while start <= end:

        week_end = min(
            start + timedelta(days=6),
            end
        )

        yield (
            start.strftime("%m-%d-%Y"),
            week_end.strftime("%m-%d-%Y")
        )

        start = week_end + timedelta(days=1)

In [ ]:
def download_week_to_zip(task_id, year, start_date, end_date):

    year_dir = os.path.join(
        OUTPUT_DIR,
        str(year)
    )

    os.makedirs(
        year_dir,
        exist_ok=True
    )


    zip_name = (
        f"ECO_L2T_LSTE_"
        f"{start_date}_"
        f"{end_date}.zip"
    )


    zip_path = os.path.join(
        year_dir,
        zip_name
    )


    bundle = requests.get(
        f"{API}bundle/{task_id}",
        headers=headers
    ).json()


    files = bundle["files"]


    with zipfile.ZipFile(
        zip_path,
        "w",
        zipfile.ZIP_DEFLATED
    ) as z:


        for file in tqdm(
            files,
            desc=f"{start_date} → {end_date}",
            leave=False
        ):

            file_id = file["file_id"]

            filename = os.path.basename(
                file["file_name"]
            )


            r = requests.get(
                f"{API}bundle/{task_id}/{file_id}",
                headers=headers,
                stream=True
            )


            r.raise_for_status()


            with z.open(filename, "w") as output:

                for chunk in r.iter_content(
                    chunk_size=1024*1024
                ):

                    if chunk:
                        output.write(chunk)


    return zip_path

In [ ]:
def process_week(args):

    year, start_date, end_date = args


    print(
        f"\nSubmitting {start_date} → {end_date}"
    )


    task_id = submit_appeears_request(
        start_date,
        end_date
    )


    wait_for_task(task_id)


    zip_path = download_week_to_zip(
        task_id,
        year,
        start_date,
        end_date
    )


    return zip_path

In [ ]:
# # TEST RUN: one week only
#
# TEST_YEAR = 2018
#
# START_DATE = "01-01-2018"
# END_DATE = "01-07-2018"
#
# print("===================")
# print("TEST RUN")
# print("===================")
#
# print(
#     f"Requesting {START_DATE} → {END_DATE}"
# )
#
# # Submit request
# task_id = submit_appeears_request(
#     START_DATE,
#     END_DATE
# )
#
# print(
#     "TASK:",
#     task_id
# )
#
# # Wait until finished
# wait_for_task(task_id)
#
# # Download files
# download_bundle(
#     task_id,
#     TEST_YEAR
# )
#
# # Zip result
# zip_year(TEST_YEAR)
#
# print("TEST COMPLETE")

In [ ]:
MAX_WORKERS = 10


for year in range(2018, 2026):


    print("\n===================")
    print(year)
    print("===================")



    if year == 2018:
        start = datetime(2018,7,10)
    else:
        start = datetime(year,1,1)


    end = datetime(year,12,31)


    weeks = []


    current = start


    while current <= end:


        week_end = min(
            current + timedelta(days=6),
            end
        )


        weeks.append(
            (
                year,
                current.strftime("%m-%d-%Y"),
                week_end.strftime("%m-%d-%Y")
            )
        )


        current = week_end + timedelta(days=1)



    print(
        f"{len(weeks)} weekly downloads"
    )


    with ThreadPoolExecutor(
        max_workers=MAX_WORKERS
    ) as executor:


        futures = [
            executor.submit(
                process_week,
                week
            )
            for week in weeks
        ]


        for future in tqdm(
            as_completed(futures),
            total=len(futures),
            desc=f"{year} complete"
        ):

            zip_file = future.result()

            print(
                "Saved:",
                zip_file
            )